In [1]:
# =========================================================
# TASK 16 — CELL 1
# SETUP + REPRODUCIBILITY
# =========================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------
# CV configuration
# ---------------------------------------------------------

N_SPLITS = 5

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("=" * 60)
print("TASK 16 — MODEL VALIDATION & K-FOLD")
print("=" * 60)

print("\nRandom state :", RANDOM_STATE)
print("CV strategy  : StratifiedKFold")
print("Number folds :", N_SPLITS)
print("Shuffle      : True")

print("\nSetup completed successfully.")

TASK 16 — MODEL VALIDATION & K-FOLD

Random state : 42
CV strategy  : StratifiedKFold
Number folds : 5
Shuffle      : True

Setup completed successfully.


In [2]:
# =========================================================
# TASK 16 — CELL 2
# LOAD IRIS DATA + PREPARE X AND Y
# =========================================================

from src.data import load_data

# Load original Iris dataset
df = load_data()

print("========== DATASET ==========")

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ---------------------------------------------------------
# Separate features and target
# ---------------------------------------------------------

X = df.drop(columns=["target"])
y = df["target"]

print("\n========== FEATURES ==========")

print("Feature shape:", X.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\n========== TARGET ==========")

print("Target shape:", y.shape)

print("\nTarget distribution:")
print(
    y.value_counts()
    .sort_index()
)

print("\nTarget classes:")
print(
    sorted(y.unique())
)

print("\nData preparation completed successfully.")

========== DATASET ==========
Dataset shape: (150, 5)

Columns:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target']

========== FEATURES ==========
Feature shape: (150, 4)

Features:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

========== TARGET ==========
Target shape: (150,)

Target distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64

Target classes:
[np.int64(0), np.int64(1), np.int64(2)]

Data preparation completed successfully.


In [3]:
# =========================================================
# TASK 16 — CELL 3
# DEFINE CANDIDATE MODELS + PREPROCESSING PIPELINES
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# ---------------------------------------------------------
# Candidate models
# ---------------------------------------------------------

models = {

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                random_state=RANDOM_STATE,
                max_iter=500
            )
        )
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=5
            )
        )
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                random_state=RANDOM_STATE
            )
        )
    ])
}

print("========== CANDIDATE MODELS ==========")

for model_name, model in models.items():

    print(f"\n{model_name}")
    print(model)

print("\nTotal candidate models:", len(models))
print("\nCandidate model setup completed successfully.")

========== CANDIDATE MODELS ==========

Logistic Regression
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=500, random_state=42))])

KNN
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])

Decision Tree
DecisionTreeClassifier(random_state=42)

Random Forest
RandomForestClassifier(n_jobs=-1, random_state=42)

SVM
Pipeline(steps=[('scaler', StandardScaler()), ('model', SVC(random_state=42))])

Total candidate models: 5

Candidate model setup completed successfully.


In [4]:
# =========================================================
# TASK 16 — CELL 4
# STRATIFIED K-FOLD DISTRIBUTION CHECK
# =========================================================

print("========== STRATIFIED K-FOLD CHECK ==========")

fold_distribution = []

for fold_number, (train_index, val_index) in enumerate(
    cv.split(X, y),
    start=1
):

    y_train_fold = y.iloc[train_index]
    y_val_fold = y.iloc[val_index]

    fold_distribution.append({
        "Fold": fold_number,
        "Train Samples": len(train_index),
        "Validation Samples": len(val_index),
        "Train Class 0": int((y_train_fold == 0).sum()),
        "Train Class 1": int((y_train_fold == 1).sum()),
        "Train Class 2": int((y_train_fold == 2).sum()),
        "Validation Class 0": int((y_val_fold == 0).sum()),
        "Validation Class 1": int((y_val_fold == 1).sum()),
        "Validation Class 2": int((y_val_fold == 2).sum())
    })

fold_distribution_df = pd.DataFrame(
    fold_distribution
)

print(
    fold_distribution_df.to_string(index=False)
)

print("\n========== FOLD SUMMARY ==========")

print(
    "Total folds:",
    fold_distribution_df["Fold"].nunique()
)

print(
    "Validation samples per fold:",
    sorted(
        fold_distribution_df[
            "Validation Samples"
        ].unique()
    )
)

print(
    "\nStratified fold verification completed successfully."
)

========== STRATIFIED K-FOLD CHECK ==========
 Fold  Train Samples  Validation Samples  Train Class 0  Train Class 1  Train Class 2  Validation Class 0  Validation Class 1  Validation Class 2
    1            120                  30             40             40             40                  10                  10                  10
    2            120                  30             40             40             40                  10                  10                  10
    3            120                  30             40             40             40                  10                  10                  10
    4            120                  30             40             40             40                  10                  10                  10
    5            120                  30             40             40             40                  10                  10                  10

========== FOLD SUMMARY ==========
Total folds: 5
Validation samples per fold

In [5]:
# =========================================================
# TASK 16 — CELL 5
# RUN CROSS-VALIDATION FOR ALL CANDIDATE MODELS
# =========================================================

cv_results = []

print("========== CROSS-VALIDATION RESULTS ==========")

for model_name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    print(f"\n{model_name}")
    print("Fold scores:", np.round(scores, 4))

    cv_results.append({
        "Model": model_name,
        "Fold_1": scores[0],
        "Fold_2": scores[1],
        "Fold_3": scores[2],
        "Fold_4": scores[3],
        "Fold_5": scores[4],
        "Mean_Accuracy": scores.mean(),
        "Std_Accuracy": scores.std(),
        "Variance_Accuracy": scores.var()
    })

cv_results_df = pd.DataFrame(cv_results)

print("\n========== CROSS-VALIDATION TABLE ==========")

print(
    cv_results_df.round(4).to_string(index=False)
)

print(
    "\nCross-validation completed successfully."
)

========== CROSS-VALIDATION RESULTS ==========

Logistic Regression
Fold scores: [1.     0.9667 0.9    1.     0.9   ]

KNN
Fold scores: [1.     0.9667 0.9333 1.     0.9667]

Decision Tree
Fold scores: [1.     0.9667 0.9333 0.9667 0.9   ]

Random Forest
Fold scores: [0.9667 0.9667 0.9333 0.9667 0.9   ]

SVM
Fold scores: [1.     0.9667 0.9    1.     0.9333]

========== CROSS-VALIDATION TABLE ==========
              Model  Fold_1  Fold_2  Fold_3  Fold_4  Fold_5  Mean_Accuracy  Std_Accuracy  Variance_Accuracy
Logistic Regression  1.0000  0.9667  0.9000  1.0000  0.9000         0.9533        0.0452             0.0020
                KNN  1.0000  0.9667  0.9333  1.0000  0.9667         0.9733        0.0249             0.0006
      Decision Tree  1.0000  0.9667  0.9333  0.9667  0.9000         0.9533        0.0340             0.0012
      Random Forest  0.9667  0.9667  0.9333  0.9667  0.9000         0.9467        0.0267             0.0007
                SVM  1.0000  0.9667  0.9000  1.0000  0.9

In [6]:
# =========================================================
# TASK 16 — CELL 6
# DETAILED CROSS-VALIDATION STATISTICS
# =========================================================

# Create a clean comparison table
model_comparison = cv_results_df[
    [
        "Model",
        "Mean_Accuracy",
        "Std_Accuracy",
        "Variance_Accuracy"
    ]
].copy()

# Rank primarily by mean accuracy,
# then by lower standard deviation.
model_comparison = model_comparison.sort_values(
    by=["Mean_Accuracy", "Std_Accuracy"],
    ascending=[False, True]
).reset_index(drop=True)

# Add generalisation rank
model_comparison["Generalisation_Rank"] = (
    np.arange(1, len(model_comparison) + 1)
)

print("========== MODEL GENERALISATION COMPARISON ==========")

print(
    model_comparison.round(4).to_string(index=False)
)

# ---------------------------------------------------------
# Best model
# ---------------------------------------------------------

best_model_name = model_comparison.iloc[0]["Model"]

best_mean_accuracy = model_comparison.iloc[0][
    "Mean_Accuracy"
]

best_std_accuracy = model_comparison.iloc[0][
    "Std_Accuracy"
]

best_variance = model_comparison.iloc[0][
    "Variance_Accuracy"
]

print("\n========== GENERALISATION WINNER ==========")

print("Selected model       :", best_model_name)
print(
    f"Mean accuracy        : "
    f"{best_mean_accuracy:.4f}"
)
print(
    f"Standard deviation   : "
    f"{best_std_accuracy:.4f}"
)
print(
    f"Variance             : "
    f"{best_variance:.4f}"
)

print(
    "\nModel selected based on highest "
    "mean cross-validation accuracy "
    "with stability considered."
)

========== MODEL GENERALISATION COMPARISON ==========
              Model  Mean_Accuracy  Std_Accuracy  Variance_Accuracy  Generalisation_Rank
                KNN         0.9733        0.0249             0.0006                    1
                SVM         0.9600        0.0389             0.0015                    2
      Decision Tree         0.9533        0.0340             0.0012                    3
Logistic Regression         0.9533        0.0452             0.0020                    4
      Random Forest         0.9467        0.0267             0.0007                    5

========== GENERALISATION WINNER ==========
Selected model       : KNN
Mean accuracy        : 0.9733
Standard deviation   : 0.0249
Variance             : 0.0006

Model selected based on highest mean cross-validation accuracy with stability considered.


In [7]:
# =========================================================
# TASK 16 — CELL 7
# INDEPENDENT TEST-SET EVALUATION
# =========================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# Create independent test set
# ---------------------------------------------------------

X_train_final, X_test_final, y_train_final, y_test_final = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=RANDOM_STATE
    )
)

print("========== FINAL TRAIN / TEST SPLIT ==========")

print("Training samples:", X_train_final.shape[0])
print("Test samples    :", X_test_final.shape[0])

print("\nTest target distribution:")
print(
    y_test_final.value_counts()
    .sort_index()
)

# ---------------------------------------------------------
# Train and evaluate every candidate model
# ---------------------------------------------------------

test_results = []

for model_name, model in models.items():

    model.fit(
        X_train_final,
        y_train_final
    )

    test_predictions = model.predict(
        X_test_final
    )

    test_accuracy = accuracy_score(
        y_test_final,
        test_predictions
    )

    test_results.append({
        "Model": model_name,
        "Test_Accuracy": test_accuracy
    })

test_results_df = pd.DataFrame(
    test_results
)

print("\n========== INDEPENDENT TEST RESULTS ==========")

print(
    test_results_df
    .sort_values(
        "Test_Accuracy",
        ascending=False
    )
    .round(4)
    .to_string(index=False)
)

print(
    "\nIndependent test evaluation completed successfully."
)

========== FINAL TRAIN / TEST SPLIT ==========
Training samples: 120
Test samples    : 30

Test target distribution:
target
0    10
1    10
2    10
Name: count, dtype: int64

========== INDEPENDENT TEST RESULTS ==========
              Model  Test_Accuracy
                SVM         0.9667
Logistic Regression         0.9333
                KNN         0.9333
      Decision Tree         0.9333
      Random Forest         0.9000

Independent test evaluation completed successfully.


In [8]:
# =========================================================
# TASK 16 — CELL 8
# FINAL MODEL SELECTION + VALIDATION SUMMARY
# =========================================================

# ---------------------------------------------------------
# Retrieve CV winner
# ---------------------------------------------------------

cv_winner = model_comparison.iloc[0]

selected_model_name = cv_winner["Model"]
selected_cv_accuracy = cv_winner["Mean_Accuracy"]
selected_cv_std = cv_winner["Std_Accuracy"]
selected_cv_variance = cv_winner["Variance_Accuracy"]

# ---------------------------------------------------------
# Retrieve test performance of CV winner
# ---------------------------------------------------------

selected_test_row = test_results_df[
    test_results_df["Model"] == selected_model_name
].iloc[0]

selected_test_accuracy = (
    selected_test_row["Test_Accuracy"]
)

# ---------------------------------------------------------
# Identify best test performer
# ---------------------------------------------------------

best_test_row = (
    test_results_df
    .sort_values(
        "Test_Accuracy",
        ascending=False
    )
    .iloc[0]
)

best_test_model = best_test_row["Model"]
best_test_accuracy = best_test_row[
    "Test_Accuracy"
]

# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print("=" * 65)
print("        TASK 16 — FINAL MODEL VALIDATION SUMMARY")
print("=" * 65)

print("\n========== CROSS-VALIDATION WINNER ==========")

print(
    "Selected model       :",
    selected_model_name
)

print(
    f"Mean CV accuracy     : "
    f"{selected_cv_accuracy:.4f}"
)

print(
    f"CV standard deviation: "
    f"{selected_cv_std:.4f}"
)

print(
    f"CV variance          : "
    f"{selected_cv_variance:.4f}"
)

print("\n========== INDEPENDENT TEST ==========")

print(
    f"{selected_model_name} test accuracy: "
    f"{selected_test_accuracy:.4f}"
)

print("\n========== BEST SINGLE TEST RESULT ==========")

print(
    "Best test model      :",
    best_test_model
)

print(
    f"Best test accuracy   : "
    f"{best_test_accuracy:.4f}"
)

print("\n========== FINAL CONCLUSION ==========")

print(
    f"{selected_model_name} was selected based on "
    "the strongest cross-validation performance."
)

print(
    f"It achieved a mean CV accuracy of "
    f"{selected_cv_accuracy:.4f} with a standard "
    f"deviation of {selected_cv_std:.4f}."
)

print(
    f"On the independent test set, the selected model "
    f"achieved {selected_test_accuracy:.4f} accuracy."
)

if best_test_model != selected_model_name:

    print(
        f"\nNote: {best_test_model} achieved the highest "
        f"accuracy ({best_test_accuracy:.4f}) on this "
        "single holdout test split."
    )

    print(
        "This does not replace the cross-validation-based "
        "model selection because the test set is an "
        "independent final evaluation set."
    )

print("\nFinal model selection completed successfully.")

        TASK 16 — FINAL MODEL VALIDATION SUMMARY

========== CROSS-VALIDATION WINNER ==========
Selected model       : KNN
Mean CV accuracy     : 0.9733
CV standard deviation: 0.0249
CV variance          : 0.0006

========== INDEPENDENT TEST ==========
KNN test accuracy: 0.9333

========== BEST SINGLE TEST RESULT ==========
Best test model      : SVM
Best test accuracy   : 0.9667

========== FINAL CONCLUSION ==========
KNN was selected based on the strongest cross-validation performance.
It achieved a mean CV accuracy of 0.9733 with a standard deviation of 0.0249.
On the independent test set, the selected model achieved 0.9333 accuracy.

Note: SVM achieved the highest accuracy (0.9667) on this single holdout test split.
This does not replace the cross-validation-based model selection because the test set is an independent final evaluation set.

Final model selection completed successfully.


In [9]:
# =========================================================
# TASK 16 — CELL 9
# SAVE VALIDATION RESULTS + ARTIFACTS
# =========================================================

import joblib

# ---------------------------------------------------------
# Create artifact directory
# ---------------------------------------------------------

artifacts_dir = PROJECT_ROOT / "artifacts"
artifacts_dir.mkdir(
    parents=True,
    exist_ok=True
)

models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Save CV comparison
# ---------------------------------------------------------

cv_results_path = (
    artifacts_dir /
    "task16_cross_validation_results.csv"
)

cv_results_df.to_csv(
    cv_results_path,
    index=False
)

# ---------------------------------------------------------
# Save model comparison/ranking
# ---------------------------------------------------------

comparison_path = (
    artifacts_dir /
    "task16_model_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

# ---------------------------------------------------------
# Save independent test results
# ---------------------------------------------------------

test_results_path = (
    artifacts_dir /
    "task16_independent_test_results.csv"
)

test_results_df.to_csv(
    test_results_path,
    index=False
)

# ---------------------------------------------------------
# Save fold distribution
# ---------------------------------------------------------

fold_distribution_path = (
    artifacts_dir /
    "task16_fold_distribution.csv"
)

fold_distribution_df.to_csv(
    fold_distribution_path,
    index=False
)

# ---------------------------------------------------------
# Save final validation summary
# ---------------------------------------------------------

final_validation_summary = pd.DataFrame([{
    "Selected_Model": selected_model_name,
    "Mean_CV_Accuracy": selected_cv_accuracy,
    "CV_Std": selected_cv_std,
    "CV_Variance": selected_cv_variance,
    "Selected_Model_Test_Accuracy": selected_test_accuracy,
    "Best_Test_Model": best_test_model,
    "Best_Test_Accuracy": best_test_accuracy,
    "CV_Strategy": "StratifiedKFold",
    "Number_of_Folds": N_SPLITS,
    "Random_State": RANDOM_STATE
}])

summary_path = (
    artifacts_dir /
    "task16_final_validation_summary.csv"
)

final_validation_summary.to_csv(
    summary_path,
    index=False
)

# ---------------------------------------------------------
# Save selected model trained on the full dataset
# ---------------------------------------------------------

selected_model = models[selected_model_name]

selected_model.fit(
    X,
    y
)

selected_model_path = (
    models_dir /
    "task16_selected_model.pkl"
)

joblib.dump(
    selected_model,
    selected_model_path
)

# ---------------------------------------------------------
# Print saved artifacts
# ---------------------------------------------------------

print("========== TASK 16 ARTIFACTS ==========")

print("\nCross-validation results:")
print(cv_results_path)

print("\nModel comparison:")
print(comparison_path)

print("\nIndependent test results:")
print(test_results_path)

print("\nFold distribution:")
print(fold_distribution_path)

print("\nFinal validation summary:")
print(summary_path)

print("\nSelected model:")
print(selected_model_path)

print("\nAll Task 16 artifacts saved successfully.")

========== TASK 16 ARTIFACTS ==========

Cross-validation results:
/home/akash/Projects/Altrodav/artifacts/task16_cross_validation_results.csv

Model comparison:
/home/akash/Projects/Altrodav/artifacts/task16_model_comparison.csv

Independent test results:
/home/akash/Projects/Altrodav/artifacts/task16_independent_test_results.csv

Fold distribution:
/home/akash/Projects/Altrodav/artifacts/task16_fold_distribution.csv

Final validation summary:
/home/akash/Projects/Altrodav/artifacts/task16_final_validation_summary.csv

Selected model:
/home/akash/Projects/Altrodav/models/task16_selected_model.pkl

All Task 16 artifacts saved successfully.


In [10]:
# =========================================================
# TASK 16 — CELL 10
# FINAL VERIFICATION
# =========================================================

import joblib
import pandas as pd
import numpy as np

print("=" * 65)
print("        TASK 16 — FINAL VERIFICATION")
print("=" * 65)

# ---------------------------------------------------------
# Load saved artifacts
# ---------------------------------------------------------

loaded_model = joblib.load(
    selected_model_path
)

loaded_cv_results = pd.read_csv(
    cv_results_path
)

loaded_comparison = pd.read_csv(
    comparison_path
)

loaded_test_results = pd.read_csv(
    test_results_path
)

loaded_fold_distribution = pd.read_csv(
    fold_distribution_path
)

loaded_summary = pd.read_csv(
    summary_path
)

print("\nSaved model loading          : PASS")
print("CV results loading           : PASS")
print("Model comparison loading     : PASS")
print("Test results loading         : PASS")
print("Fold distribution loading    : PASS")
print("Final summary loading        : PASS")

# ---------------------------------------------------------
# Verify selected model
# ---------------------------------------------------------

print("\n========== SELECTED MODEL ==========")

print(
    "Selected model:",
    selected_model_name
)

print(
    "Saved model type:",
    type(loaded_model).__name__
)

# ---------------------------------------------------------
# Verify predictions
# ---------------------------------------------------------

loaded_predictions = loaded_model.predict(
    X
)

prediction_count = len(
    loaded_predictions
)

print(
    "\nPredictions generated:",
    prediction_count
)

prediction_check = (
    prediction_count == len(X)
)

print(
    "Prediction count check:",
    "PASS" if prediction_check else "FAIL"
)

# ---------------------------------------------------------
# Verify CV winner
# ---------------------------------------------------------

loaded_cv_winner = (
    loaded_comparison
    .sort_values(
        by=[
            "Mean_Accuracy",
            "Std_Accuracy"
        ],
        ascending=[
            False,
            True
        ]
    )
    .iloc[0]
)

cv_winner_check = (
    loaded_cv_winner["Model"]
    == selected_model_name
)

print(
    "CV winner verification:",
    "PASS" if cv_winner_check else "FAIL"
)

# ---------------------------------------------------------
# Verify fold structure
# ---------------------------------------------------------

fold_check = (
    len(loaded_fold_distribution) == N_SPLITS
    and
    loaded_fold_distribution[
        "Validation Samples"
    ].eq(30).all()
)

print(
    "Stratified fold verification:",
    "PASS" if fold_check else "FAIL"
)

# ---------------------------------------------------------
# Verify final metrics
# ---------------------------------------------------------

print("\n========== FINAL METRICS ==========")

print(
    f"Mean CV Accuracy : "
    f"{selected_cv_accuracy:.4f}"
)

print(
    f"CV Std           : "
    f"{selected_cv_std:.4f}"
)

print(
    f"CV Variance      : "
    f"{selected_cv_variance:.4f}"
)

print(
    f"KNN Test Accuracy: "
    f"{selected_test_accuracy:.4f}"
)

print(
    f"Best Test Model  : "
    f"{best_test_model}"
)

print(
    f"Best Test Accuracy: "
    f"{best_test_accuracy:.4f}"
)

# ---------------------------------------------------------
# Final verification status
# ---------------------------------------------------------

all_checks_passed = (
    prediction_check
    and cv_winner_check
    and fold_check
    and len(loaded_cv_results) == 5
    and len(loaded_test_results) == 5
)

print("\n" + "=" * 65)

if all_checks_passed:
    print("TASK 16 COMPLETED SUCCESSFULLY!")
else:
    print(
        "TASK 16 VERIFICATION FAILED — "
        "REVIEW RESULTS"
    )

print("=" * 65)

        TASK 16 — FINAL VERIFICATION

Saved model loading          : PASS
CV results loading           : PASS
Model comparison loading     : PASS
Test results loading         : PASS
Fold distribution loading    : PASS
Final summary loading        : PASS

========== SELECTED MODEL ==========
Selected model: KNN
Saved model type: Pipeline

Predictions generated: 150
Prediction count check: PASS
CV winner verification: PASS
Stratified fold verification: PASS

========== FINAL METRICS ==========
Mean CV Accuracy : 0.9733
CV Std           : 0.0249
CV Variance      : 0.0006
KNN Test Accuracy: 0.9333
Best Test Model  : SVM
Best Test Accuracy: 0.9667

TASK 16 COMPLETED SUCCESSFULLY!
